# 20 · BirdNET vs. ResNet152V2 — Inferencia de BirdNET sobre el test set

Este notebook aplica **BirdNET V2.4** al conjunto de test del proyecto para compararlo
con el modelo propio (**ResNet152V2**, 667 especies). Como ya no se conservan los audios
originales —solo los espectrogramas—, se **re-descargan las grabaciones originales de
Xeno-canto** usando el ID que quedó codificado en el nombre de cada imagen
(`<xc_id>_<offset>.jpeg`, ver `src/spectograms.py`).

Decisiones de diseño (acordadas):
- **Comparación por grabación (top-1)**: BirdNET segmenta en ventanas de 3 s; se agregan a
  una predicción por grabación.
- **Espacio de clases restringido** a las especies del dataset que BirdNET conoce
  (`custom_species_list`) → comparación justa.
- **Muestra estratificada** (N grabaciones por especie) para una primera corrida manejable.

> **Kernel**: ejecuta este notebook con el entorno aislado **`.venv-birdnet`** (Python 3.11 +
> `birdnet`), NO con el `.venv` del proyecto (que tiene TensorFlow 2.19 para el ResNet).

Salidas (en `src/data/`): `birdnet_species_coverage.csv`, `birdnet_sample_recordings.csv`,
`birdnet_segments.csv`, `birdnet_predictions.csv`.

## Requisitos

**1. Entorno aislado con BirdNET** (ya creado si seguiste el plan):
```bash
uv venv .venv-birdnet --python 3.11
uv pip install --python .venv-birdnet/bin/python --index-url https://pypi.org/simple birdnet requests scikit-learn ipykernel
.venv-birdnet/bin/python -m ipykernel install --user --name birdnet --display-name "Python (birdnet)"
```
Selecciona el kernel **Python (birdnet)** para este notebook.

**2. Sin API key**: los audios se descargan por la URL pública de Xeno-canto
(`https://xeno-canto.org/<id>/download`), que **no requiere clave** (el mismo acceso
que usaba `xenopy`). Solo hace falta conexión a internet.

In [1]:
import os, sys, multiprocessing as mp

# BirdNET usa multiprocessing internamente. En macOS el método por defecto es 'spawn',
# que no funciona dentro de Jupyter; forzamos 'fork'.
try:
    mp.set_start_method("fork", force=True)
except RuntimeError:
    pass

import numpy as np
import pandas as pd

sys.path.append("../src")
import birdnet_utils as bu

# ---------------- Configuración ----------------
ROOT        = os.path.abspath("..")
IMAGES_ROOT = os.path.join(ROOT, "src/data/images_test/images_spectograms")
DATA_DIR    = os.path.join(ROOT, "src/data")
AUDIO_DIR   = os.path.join(ROOT, "data/audio_test_sample")

# N_PER_SPECIES: grabaciones por especie a evaluar.
#   entero (p. ej. 2) -> muestra rápida  |  None -> TODO el test (~18k audios:
#   la 1ª corrida tarda por descarga+inferencia, pero es reanudable por checkpointing).
N_PER_SPECIES = 2
HOW_AGG       = "max"  # agregación de segmentos -> grabación: 'max' o 'mean'
TOP_K         = 5      # nº de especies por segmento que devuelve BirdNET
SEED          = 42

# Los audios se descargan por la URL pública de Xeno-canto (sin API key).
os.makedirs(AUDIO_DIR, exist_ok=True)
print("images :", IMAGES_ROOT)
print("audio  :", AUDIO_DIR)

images : /Users/camcortes/Documents/birds-sounds/src/data/images_test/images_spectograms
audio  : /Users/camcortes/Documents/birds-sounds/data/audio_test_sample


In [2]:
# 1) Índice del test set a nivel de grabación (a partir de los espectrogramas)
index = bu.build_test_index(IMAGES_ROOT)
print("grabaciones:", index.recording_id.nunique(),
      "| especies:", index.species.nunique(),
      "| chunks:", int(index.n_chunks.sum()))
index.head()

grabaciones: 18227 | especies: 667 | chunks: 32279


,recording_id,species,n_chunks,chunk_paths
0,104508,Acropternis orthonyx,1,[/Users/camcortes/Documents/birds-sounds/src/d...
1,119693,Acropternis orthonyx,1,[/Users/camcortes/Documents/birds-sounds/src/d...
2,119694,Acropternis orthonyx,2,[/Users/camcortes/Documents/birds-sounds/src/d...
3,121142,Acropternis orthonyx,3,[/Users/camcortes/Documents/birds-sounds/src/d...
4,127718,Acropternis orthonyx,1,[/Users/camcortes/Documents/birds-sounds/src/d...


In [3]:
# 2) Cargar BirdNET V2.4 y calcular la cobertura de especies del dataset
import birdnet

model = birdnet.load("acoustic", "2.4", "tf")
print("BirdNET V2.4 |", model.n_species, "especies |",
      model.get_sample_rate(), "Hz |", model.get_segment_size_s(), "s/segmento")

# Persistir la lista de especies del modelo
species_list = list(model.species_list)
with open(os.path.join(DATA_DIR, "birdnet_v2.4_species_list.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(species_list))
labels_df = bu.birdnet_labels_to_df(species_list)

# Cobertura: cruzar las especies del dataset con la taxonomía de BirdNET
dataset_species = sorted(index.species.unique())
coverage = bu.match_species(dataset_species, labels_df)
gf = bu.get_genus_family_map()
coverage["family"] = coverage["species"].map(lambda s: bu.family_of(s, gf))
coverage.to_csv(os.path.join(DATA_DIR, "birdnet_species_coverage.csv"), index=False)

n_cov = int(coverage.covered.sum())
print(f"Cobertura BirdNET: {n_cov}/{len(coverage)} ({100*n_cov/len(coverage):.1f}%)")
print("NO cubiertas:", coverage.loc[~coverage.covered, "species"].tolist())

BirdNET V2.4 | 6522 especies | 48000 Hz | 3.0 s/segmento
Cobertura BirdNET: 664/667 (99.6%)
NO cubiertas: ['Haplospiza rustica', 'Hylopezus fulviventris', 'Myiophobus roraimae']


In [4]:
# 3) Muestra estratificada (solo especies que BirdNET puede predecir)
covered_species = set(coverage.loc[coverage.covered, "species"])
sample = bu.stratified_sample(index, N_PER_SPECIES,
                              covered_species=covered_species, seed=SEED)
sci2label = dict(zip(coverage.species, coverage.birdnet_label))
sample["birdnet_label_true"] = sample.species.map(sci2label)
print("grabaciones en la muestra:", len(sample), "| especies:", sample.species.nunique())
sample.head()

grabaciones en la muestra: 1328 | especies: 664


,recording_id,species,n_chunks,chunk_paths,birdnet_label_true
0,621780,Acropternis orthonyx,2,[/Users/camcortes/Documents/birds-sounds/src/d...,Acropternis orthonyx_Ocellated Tapaculo
1,374408,Acropternis orthonyx,4,[/Users/camcortes/Documents/birds-sounds/src/d...,Acropternis orthonyx_Ocellated Tapaculo
2,28567,Amblycercus holosericeus,1,[/Users/camcortes/Documents/birds-sounds/src/d...,Amblycercus holosericeus_Yellow-billed Cacique
3,332429,Amblycercus holosericeus,2,[/Users/camcortes/Documents/birds-sounds/src/d...,Amblycercus holosericeus_Yellow-billed Cacique
4,836981,Ammodramus aurifrons,3,[/Users/camcortes/Documents/birds-sounds/src/d...,Ammodramus aurifrons_Yellow-browed Sparrow


In [5]:
# 4) Descargar los audios originales de Xeno-canto (checkpointing: omite los ya bajados)
import requests
session = requests.Session()

paths, ok, fail = [], 0, 0
for i, row in enumerate(sample.itertuples(), 1):
    p = bu.download_xc_recording(row.recording_id, AUDIO_DIR,
                                 species=row.species, session=session)
    paths.append(p)
    ok += bool(p); fail += (not p)
    if i % 50 == 0:
        print(f"  {i}/{len(sample)}  ok={ok}  fallidos={fail}")

sample["audio_path"] = paths
sample.to_csv(os.path.join(DATA_DIR, "birdnet_sample_recordings.csv"), index=False)
print(f"Descarga terminada: OK={ok}  fallidos/eliminados={fail}  total={len(sample)}")

  50/1328  ok=50  fallidos=0
  100/1328  ok=100  fallidos=0
  150/1328  ok=150  fallidos=0
  200/1328  ok=200  fallidos=0
  250/1328  ok=250  fallidos=0
  300/1328  ok=300  fallidos=0
  350/1328  ok=350  fallidos=0
  400/1328  ok=400  fallidos=0
  450/1328  ok=450  fallidos=0
  500/1328  ok=500  fallidos=0
  550/1328  ok=550  fallidos=0
  600/1328  ok=600  fallidos=0
  650/1328  ok=650  fallidos=0
  700/1328  ok=700  fallidos=0
  750/1328  ok=750  fallidos=0
  800/1328  ok=800  fallidos=0
  850/1328  ok=850  fallidos=0
  900/1328  ok=900  fallidos=0
  950/1328  ok=950  fallidos=0
  1000/1328  ok=1000  fallidos=0
  1050/1328  ok=1050  fallidos=0
  1100/1328  ok=1100  fallidos=0
  1150/1328  ok=1150  fallidos=0
  1200/1328  ok=1200  fallidos=0
  1250/1328  ok=1250  fallidos=0
  1300/1328  ok=1300  fallidos=0
Descarga terminada: OK=1328  fallidos/eliminados=0  total=1328


In [ ]:
# 5) Inferencia BirdNET restringida a las especies del dataset (custom_species_list)
custom_species = sorted(coverage.loc[coverage.covered, "birdnet_label"].dropna().unique())
candidate_paths = [p for p in sample.audio_path.tolist() if p]

# Validar los audios: BirdNET cancela TODO el lote si un archivo está vacío o
# corrupto (algunos MP3 de Xeno-canto tienen el stream truncado).
audio_paths, descartados = [], []
for p in candidate_paths:
    (audio_paths if bu.is_valid_audio(p, min_duration_s=0.5) else descartados).append(p)
print(f"Audios válidos: {len(audio_paths)} | descartados (vacíos/corruptos): {len(descartados)} "
      f"| especies permitidas: {len(custom_species)}")
for p in descartados:
    print("   descartado:", os.path.basename(p))

# Inferencia robusta: procesa por lotes y aísla (por bisección) cualquier MP3 que
# BirdNET no pueda procesar, para que un archivo problemático no cancele todo el lote.
seg_df, failed = bu.predict_birdnet_robust(
    model, audio_paths, custom_species, top_k=TOP_K, batch_size=64,
    default_confidence_threshold=0.0,   # no descartar nada: queremos el top-1 por grabación
    on_progress=lambda done, tot: print(f"  procesados {done}/{tot}") if done % 256 == 0 or done == tot else None,
)
if failed:
    print(f"AVISO: {len(failed)} audios omitidos (BirdNET no pudo procesarlos; p. ej. MP3 con frames corruptos):")
    for p in failed:
        print("   ", os.path.basename(p))
seg_df.to_csv(os.path.join(DATA_DIR, "birdnet_segments.csv"), index=False)
print("filas (segmento x especie):", len(seg_df), "| audios con predicción:", seg_df.input.nunique())
seg_df.head()

In [ ]:
# 6) Agregar a una predicción top-1 por grabación
path2rec  = dict(zip(sample.audio_path, sample.recording_id))
path2true = dict(zip(sample.audio_path, sample.species))

rows = []
for inp, g in seg_df.groupby("input"):
    top_label, conf = bu.birdnet_top1_from_dataframe(g, how=HOW_AGG)
    rows.append({
        "recording_id": path2rec.get(inp),
        "species_true": path2true.get(inp),
        "birdnet_label": top_label,
        "species_pred": bu.birdnet_label_to_scientific(top_label),
        "confidence": conf,
    })
pred = pd.DataFrame(rows)
pred.to_csv(os.path.join(DATA_DIR, "birdnet_predictions.csv"), index=False)

n_no_pred = len(audio_paths) - pred.species_pred.notna().sum()
print("predicciones recording-level:", len(pred),
      "| audios sin predicción de BirdNET:", int(n_no_pred))
pred.head()

In [ ]:
# 7) Métricas de BirdNET sobre la muestra (recording-level)
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

valid = pred.dropna(subset=["species_pred"])
acc = accuracy_score(valid.species_true, valid.species_pred)
f1m = f1_score(valid.species_true, valid.species_pred, average="macro", zero_division=0)
prm = precision_score(valid.species_true, valid.species_pred, average="macro", zero_division=0)
rcm = recall_score(valid.species_true, valid.species_pred, average="macro", zero_division=0)

print(f"BirdNET V2.4 · recording-level · N_PER_SPECIES={N_PER_SPECIES}")
print(f"  grabaciones en la muestra : {len(sample)}")
print(f"  descargas fallidas        : {fail}")
print(f"  audios inválidos (filtro) : {len(descartados)}")
print(f"  no procesables por BirdNET: {len(failed)}")
print(f"  grabaciones evaluadas     : {len(valid)}")
print(f"  accuracy                  : {acc:.4f}")
print(f"  F1 macro                  : {f1m:.4f}")
print(f"  precision macro           : {prm:.4f}")
print(f"  recall macro              : {rcm:.4f}")
print("\nSiguiente paso: ejecuta 21_resnet_recording_eval_and_compare.ipynb (kernel .venv del proyecto).")